# Model Saving & Checkpointing — Complete Theory

---

## What Are We Actually Saving?

A trained model has two distinct things:

**1. Architecture** — the structure (layers, sizes, connections). This lives in your Python class definition.

**2. State** — the learned values (weights and biases). This is what training actually produces.

You only need to save the **state**. The architecture is already in your code. PyTorch stores all learnable parameters in a dictionary called the `state_dict` — literally a Python dict mapping layer names to their tensors.

Print yours and you'll see something like:
``` t
network.0.weight → tensor of shape (64, 5)
network.0.bias   → tensor of shape (64,)
...
```

---

## What is a Checkpoint?

A checkpoint is a snapshot of your training at a specific moment. A minimal checkpoint saves just the model weights. A full checkpoint saves everything needed to **resume training exactly where you left off**:

``` python
Checkpoint = {
    model weights       # ← to restore predictions
    optimizer state     # ← to resume weight updates correctly
    scheduler state     # ← to resume LR scheduling correctly
    epoch number        # ← to know where you stopped
    best val loss       # ← to know if new epochs improve things
    val accuracy        # ← for reference
}
```

Why save the optimizer state? Adam isn't a simple gradient scaler — it maintains momentum estimates for every parameter. If you restore weights but reset the optimizer, your first few resumed epochs will behave differently than if you'd never stopped.

---

## Saving — `torch.save()`

PyTorch uses Python's `pickle` format under the hood. The convention is `.pt` or `.pth` file extension.

**Save just weights (minimal):**
```python
torch.save(model.state_dict(), 'model.pt')
```

**Save full checkpoint (production):**
```python
checkpoint = {
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'val_loss': avg_val_loss,
    'val_acc': avg_val_acc,
}
torch.save(checkpoint, 'checkpoint.pt')
```

---

## Loading — `torch.load()` + `load_state_dict()`

Loading is two steps — load the file, then inject the state into your model:

**Load just weights:**
```python
model = SpeakerCounter().to(device)  # create architecture first
model.load_state_dict(torch.load('model.pt'))
```

**Load full checkpoint to resume training:**
```python
checkpoint = torch.load('checkpoint.pt')
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
start_epoch = checkpoint['epoch'] + 1
best_val_loss = checkpoint['val_loss']
```

**Important:** After loading for inference (not resuming training), call `model.eval()`. After loading to resume training, call `model.train()` before your loop.

---

## When to Save — Two Strategies

**Strategy 1 — Save every N epochs:** Simple, but you might save a bad checkpoint if the model degraded.

**Strategy 2 — Save only when val loss improves (best model):** This is what you want. Keep track of the best val loss seen so far. Only overwrite the checkpoint file when the current epoch beats it.

``` python
best_val_loss = infinity
if current_val_loss < best_val_loss:
    best_val_loss = current_val_loss
    save checkpoint  ← only save when it's the best so far
```

This means your saved checkpoint always contains the best model you've ever seen, regardless of what happens in later epochs.

---

## File Organization

Keep checkpoints in a dedicated folder. Two files are typical:
- `best_model.pt` — overwrites every time a new best is found
- `last_checkpoint.pt` — overwrites every epoch (for resuming if crashed)

---

## TODO 1: Save and Load Weights Only

**Goal:** Save your model weights after training, load them into a fresh model, verify predictions are identical.

**Requirements:**
- After your training loop, save `model.state_dict()` to `outputs/checkpoints/model_weights.pt`
- Create a brand new `SpeakerCounter` instance called `loaded_model`
- Load the saved weights into it
- Run both models on the same batch, compare outputs with `torch.allclose()`
- Print whether they match

**Hints:**
- `Path.mkdir(parents=True, exist_ok=True)` to create the directory
- `torch.allclose(a, b)` returns `True` if tensors are equal within floating point tolerance
- Set both models to `eval()` before comparing — BatchNorm behaves differently in train mode
- `torch.load()` on newer PyTorch needs `weights_only=True` parameter to avoid a warning

---

## TODO 2: Full Checkpoint Inside Training Loop

**Goal:** Add best-model checkpointing to your training loop.

**Requirements:**
- Initialize `best_val_loss = float('inf')` before the loop
- After each validation phase, compare `avg_val_loss` to `best_val_loss`
- If improved: save full checkpoint, update `best_val_loss`, print a message
- If not improved: print "no improvement" message
- Checkpoint should include model, optimizer, scheduler states, epoch, val loss, val acc

**Expected output:**
``` t
Epoch 1 | ... | ✅ New best! Saved checkpoint (val_loss: 1.1127)
Epoch 2 | ... | ✅ New best! Saved checkpoint (val_loss: 1.0733)
Epoch 3 | ... | ✅ New best! Saved checkpoint (val_loss: 1.0158)
Epoch 4 | ... | ⏭ No improvement
```

---

## TODO 3: Load and Resume

**Goal:** Load your checkpoint and prove you can resume training from where you stopped.

**Requirements:**
- Load the checkpoint file
- Restore model, optimizer, scheduler states
- Print the epoch it was saved at and the val loss it achieved
- Run 2 more epochs and confirm loss continues from where it left off, not from scratch



In [116]:
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from pathlib import Path
import os
import sys
import json

# Check if __file__ exists (it won't in Jupyter)
try:
    current_dir = Path(__file__).parent
except NameError:
    # If in Jupyter, use the current working directory
    current_dir = Path(os.getcwd())

# Add project root to Python path
project_root = current_dir.parent.parent.parent
sys.path.insert(0, str(project_root))

from src.preprocessing.audio_utils import load_audio


# Device configuration (for your MacBook)
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print("✅ Using Apple Silicon GPU")
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print("✅ Using NVIDIA GPU")
else:
    device = torch.device('cpu')
    print("⚠️ Using CPU")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")


✅ Using Apple Silicon GPU
PyTorch version: 2.10.0
Device: mps


### loading the sample

In [117]:
class SpeakerDataset(torch.utils.data.Dataset):
    def __init__(self, manifest_path):
        # Load manifest JSON
        # Store all entries as self.data
        with open(manifest_path, "r") as f:
            self.data = json.load(f)
    
    def __len__(self):
        # Return number of samples
        return len(self.data)
    
    def __getitem__(self, idx):
        # Get entry at index idx
        # Load audio from disk
        # Extract features
        # Get label (num_speakers - 1)
        # Return (features, label)
        entry = self.data[idx]
        mixture_path = Path(entry['mixture_path'])
        mixture_audio, _ = load_audio(mixture_path)
        
        mixture_tensor = torch.from_numpy(mixture_audio)
        mixture_tensor = mixture_tensor
        feature_tensor = self.feature_extraction(mixture_tensor)
        
        speaker_count = int(entry['num_speakers']) - 1
        label_tensor = torch.tensor(speaker_count, dtype=torch.long)
        
        return feature_tensor, label_tensor
    
    def feature_extraction(
            self,
            audio: torch.Tensor
    ) -> torch.Tensor:
        mean = torch.mean(audio)
        std = torch.std(audio)
        min = torch.min(audio)
        max = torch.max(audio)
        square = audio ** 2
        energy = torch.mean(square)
        
        return torch.stack([mean, std, min, max, energy])

### getting the path to the dataset and loading the metadata

In [118]:
train_manifest_path = Path.cwd().parent.parent.parent / "data" / "processed" / "train" / "train_manifest.json"
train_dataset = SpeakerDataset(train_manifest_path)
print(len(train_dataset))

val_manifest_path = Path.cwd().parent.parent.parent / "data" / "processed" / "val" / "val_manifest.json"
val_dataset = SpeakerDataset(val_manifest_path)
print(len(val_dataset))

test_manifest_path = Path.cwd().parent.parent.parent / "data" / "processed" / "test" / "test_manifest.json"
test_dataset = SpeakerDataset(test_manifest_path)
print(len(test_dataset))


10000
10000
10000


In [119]:
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,
    drop_last=True
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    drop_last=False
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    drop_last=False
)

### creating the neural network

In [120]:
class SpeakerCounter(nn.Module):
    """
    Classifier that predicts number of speakers (1, 2, or 3)
    from audio features.
    """
    def __init__(self, input_size=5, hidden_sizes=[64, 32, 16], num_classes=3):
        super().__init__()
        # TODO: Build network with given architecture
        # Hint: Use nn.Sequential or define layers individually
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_sizes[0]),
            nn.BatchNorm1d(hidden_sizes[0]),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden_sizes[0],hidden_sizes[1]),
            nn.BatchNorm1d(hidden_sizes[1]),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_sizes[1],hidden_sizes[2]),
            nn.BatchNorm1d(hidden_sizes[2]),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_sizes[2],num_classes)
        )
        
    def forward(self, x):
        # TODO: Implement forward pass
        return self.network(x)

### calculating the loss

In [121]:
def compute_accuracy(predictions, labels):
    """
    predictions: (batch_size, 3) logits
    labels: (batch_size,) true classes
    
    Returns: accuracy as percentage
    """
    predicted_classes = torch.argmax(predictions, dim=1)
    correct = (predicted_classes == labels).sum().item()
    total = labels.size(0)
    return 100.0 * correct / total

### training and validation loop

In [122]:
model = SpeakerCounter().to(device)

optimizer = optim.Adam(model.parameters(), lr=0.0001)
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5, min_lr=1e-6)
num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    running_train_loss = 0.0
    running_train_acc = 0.0
    steps_per_epoch = len(train_loader)
    
    for step, batch in enumerate(train_loader):
        features, labels = batch[0].to(device), batch[1].to(device)
        
        # Forward pass
        logits = model(features)  
        
        # Compute loss
        loss = F.cross_entropy(logits, labels)
        
        # Backward pass
        loss.backward()
        
        # Update weights
        optimizer.step()
        optimizer.zero_grad()
        
        # Compute accuracy
        running_train_loss += loss.item() 
        running_train_acc += compute_accuracy(logits, labels)
        
        # Optional: Print batch progress every 50 batches within the epoch
        if (step + 1) % 50 == 0:
            print(f"  train Batch {step+1}/{steps_per_epoch} | Loss: {loss.item():.4f}")
    
    # 3. Print Summary at the end of each FULL epoch
    avg_train_loss = running_train_loss / steps_per_epoch
    avg_train_acc = running_train_acc / steps_per_epoch    
    
    model.eval()
    running_val_loss = 0.0
    running_val_acc = 0.0      
    
    with torch.no_grad():
        for steps, batch in enumerate(val_loader):
            
            features, labels = batch[0].to(device), batch[1].to(device)
            
            # Forward pass
            logits = model(features)  
            
            # Compute loss
            loss = F.cross_entropy(logits, labels)
            
            # number of correct class
            running_val_loss += loss.item()
            running_val_acc += compute_accuracy(logits, labels)  
            
            # Optional: Print batch progress every 50 batches within the epoch
            if (steps + 1) % 50 == 0:
                print(f"  val Batch {steps+1}/{steps_per_epoch} | Loss: {loss.item():.4f}")
    
    # Calculate Averages
    avg_val_loss = running_val_loss / len(val_loader)
    avg_val_acc = running_val_acc / len(val_loader)    
    
    # --- LOGGING & SCHEDULER ---
    print(f"\n--- Epoch {epoch+1}/{num_epochs} Summary ---")
    print(f"Train Loss: {avg_train_loss:.4f} | Train Acc: {avg_train_acc:.2f}%")
    print(f"Val Loss:   {avg_val_loss:.4f} | Val Acc:   {avg_val_acc:.2f}%")
    
    scheduler.step(avg_val_loss)
    
    # Optional: Print current learning rate
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Current Learning Rate: {current_lr}\n")




  train Batch 50/156 | Loss: 1.0192
  train Batch 100/156 | Loss: 1.0245
  train Batch 150/156 | Loss: 1.0598
  val Batch 50/156 | Loss: 1.0292
  val Batch 100/156 | Loss: 1.0271
  val Batch 150/156 | Loss: 1.0316

--- Epoch 1/5 Summary ---
Train Loss: 1.0636 | Train Acc: 46.21%
Val Loss:   1.0187 | Val Acc:   49.99%
Current Learning Rate: 0.0001

  train Batch 50/156 | Loss: 0.9907
  train Batch 100/156 | Loss: 0.9764
  train Batch 150/156 | Loss: 0.9824
  val Batch 50/156 | Loss: 1.0001
  val Batch 100/156 | Loss: 0.9955
  val Batch 150/156 | Loss: 0.9993

--- Epoch 2/5 Summary ---
Train Loss: 1.0099 | Train Acc: 48.85%
Val Loss:   0.9887 | Val Acc:   49.83%
Current Learning Rate: 0.0001

  train Batch 50/156 | Loss: 0.9544
  train Batch 100/156 | Loss: 0.9646
  train Batch 150/156 | Loss: 0.9360
  val Batch 50/156 | Loss: 0.9635
  val Batch 100/156 | Loss: 0.9593
  val Batch 150/156 | Loss: 0.9643

--- Epoch 3/5 Summary ---
Train Loss: 0.9604 | Train Acc: 50.27%
Val Loss:   0.9541 |

### todo 1

In [123]:
project_root = current_dir.parent.parent.parent
checkpoint_save1 = project_root / "notebooks" / "phase2notes" / "2_4_training_minior_topics" / "checkpoints" / "todo1"
checkpoint_save1.mkdir(parents=True, exist_ok=True)
checkpoint_save1 = checkpoint_save1 / "checkpoint.pt"

In [124]:
checkpoint = {
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'val_loss': avg_val_loss,
    'val_acc': avg_val_acc,
}
torch.save(checkpoint, checkpoint_save1)



In [125]:
loaded_model1 = SpeakerCounter().to(device)

checkpoint_loaded1 = torch.load(checkpoint_save1)

loaded_model1.load_state_dict(checkpoint_loaded1['model_state_dict'])

<All keys matched successfully>

In [126]:
# 1. Set the model to evaluation mode
model.eval()

# 2. Grab a single batch from the validation loader
# iter() creates an iterator, and next() grabs the first item
features, labels = next(iter(val_loader))

# 3. Move data to the same device as the model
features = features.to(device)
labels = labels.to(device)

# 4. Run the forward pass without tracking gradients
with torch.no_grad():
    logits = model(features)
    
    # Calculate accuracy using your helper function
    batch_acc = compute_accuracy(logits, labels)
    
    # Get the actual class predictions (0, 1, or 2)
    a = torch.argmax(logits, dim=1)

# 5. Print the results
print(f"Batch Accuracy: {batch_acc:.2f}%")
print(f"Predicted classes: {a[:10].tolist()}...") # Show first 10
print(f"True labels:      {labels[:10].tolist()}...")

Batch Accuracy: 50.00%
Predicted classes: [2, 1, 1, 1, 2, 1, 1, 2, 1, 1]...
True labels:      [1, 1, 1, 1, 2, 2, 1, 1, 1, 1]...


In [127]:
# 1. Set the model to evaluation mode
loaded_model1.eval()

# 2. Grab a single batch from the validation loader
# iter() creates an iterator, and next() grabs the first item
features, labels = next(iter(val_loader))

# 3. Move data to the same device as the model
features = features.to(device)
labels = labels.to(device)

# 4. Run the forward pass without tracking gradients
with torch.no_grad():
    logits = loaded_model1(features)
    
    # Calculate accuracy using your helper function
    batch_acc = compute_accuracy(logits, labels)
    
    # Get the actual class predictions (0, 1, or 2)
    b = torch.argmax(logits, dim=1)

# 5. Print the results
print(f"Batch Accuracy: {batch_acc:.2f}%")
print(f"Predicted classes: {a[:10].tolist()}...") # Show first 10
print(f"True labels:      {labels[:10].tolist()}...")

Batch Accuracy: 50.00%
Predicted classes: [2, 1, 1, 1, 2, 1, 1, 2, 1, 1]...
True labels:      [1, 1, 1, 1, 2, 2, 1, 1, 1, 1]...


In [128]:
torch.allclose(a, b)

True

### todo 2

In [129]:
project_root = current_dir.parent.parent.parent
checkpoint_save2 = project_root / "notebooks" / "phase2notes" / "2_4_training_minior_topics" / "checkpoints" / "todo2"
checkpoint_save2.mkdir(parents=True, exist_ok=True)
checkpoint_save2 = checkpoint_save2 / "checkpoint.pt"

In [130]:
model = SpeakerCounter().to(device)

optimizer = optim.Adam(model.parameters(), lr=0.0001)
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5, min_lr=1e-6)
best_val_loss = float('inf')
num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    running_train_loss = 0.0
    running_train_acc = 0.0
    steps_per_epoch = len(train_loader)
    
    for step, batch in enumerate(train_loader):
        features, labels = batch[0].to(device), batch[1].to(device)
        
        # Forward pass
        logits = model(features)  
        
        # Compute loss
        loss = F.cross_entropy(logits, labels)
        
        # Backward pass
        loss.backward()
        
        # Update weights
        optimizer.step()
        optimizer.zero_grad()
        
        # Compute accuracy
        running_train_loss += loss.item() 
        running_train_acc += compute_accuracy(logits, labels)
        
        # Optional: Print batch progress every 50 batches within the epoch
        if (step + 1) % 50 == 0:
            print(f"  train Batch {step+1}/{steps_per_epoch} | Loss: {loss.item():.4f}")
    
    # 3. Print Summary at the end of each FULL epoch
    avg_train_loss = running_train_loss / steps_per_epoch
    avg_train_acc = running_train_acc / steps_per_epoch    
    
    model.eval()
    running_val_loss = 0.0
    running_val_acc = 0.0      
    
    with torch.no_grad():
        for steps, batch in enumerate(val_loader):
            
            features, labels = batch[0].to(device), batch[1].to(device)
            
            # Forward pass
            logits = model(features)  
            
            # Compute loss
            loss = F.cross_entropy(logits, labels)
            
            # number of correct class
            running_val_loss += loss.item()
            running_val_acc += compute_accuracy(logits, labels)  
            
            # Optional: Print batch progress every 50 batches within the epoch
            if (steps + 1) % 50 == 0:
                print(f"  val Batch {steps+1}/{steps_per_epoch} | Loss: {loss.item():.4f}")
    
    # Calculate Averages
    avg_val_loss = running_val_loss / len(val_loader)
    avg_val_acc = running_val_acc / len(val_loader)    
    
    # --- LOGGING & SCHEDULER ---
    print(f"\n--- Epoch {epoch+1}/{num_epochs} Summary ---")
    print(f"Train Loss: {avg_train_loss:.4f} | Train Acc: {avg_train_acc:.2f}%")
    print(f"Val Loss:   {avg_val_loss:.4f} | Val Acc:   {avg_val_acc:.2f}%")
    
    scheduler.step(avg_val_loss)
    
    # Optional: Print current learning rate
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Current Learning Rate: {current_lr}\n")
    
    if avg_val_loss < best_val_loss:
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_loss': avg_val_loss,
            'val_acc': avg_val_acc,
        }
        torch.save(checkpoint, checkpoint_save2)
        print(f"new best! Saved CheckPoint (loss:{avg_val_loss:.6f})")
    else:
        print("No improvement")




  train Batch 50/156 | Loss: 1.2617
  train Batch 100/156 | Loss: 1.2583
  train Batch 150/156 | Loss: 1.1509
  val Batch 50/156 | Loss: 1.0815
  val Batch 100/156 | Loss: 1.1030
  val Batch 150/156 | Loss: 1.1069

--- Epoch 1/5 Summary ---
Train Loss: 1.2405 | Train Acc: 26.26%
Val Loss:   1.1150 | Val Acc:   48.86%
Current Learning Rate: 0.0001

new best! Saved CheckPoint (loss:1.115003)
  train Batch 50/156 | Loss: 1.1550
  train Batch 100/156 | Loss: 1.1542
  train Batch 150/156 | Loss: 1.1635
  val Batch 50/156 | Loss: 1.0311
  val Batch 100/156 | Loss: 1.0604
  val Batch 150/156 | Loss: 1.0615

--- Epoch 2/5 Summary ---
Train Loss: 1.1445 | Train Acc: 36.64%
Val Loss:   1.0666 | Val Acc:   49.73%
Current Learning Rate: 0.0001

new best! Saved CheckPoint (loss:1.066606)
  train Batch 50/156 | Loss: 1.0667
  train Batch 100/156 | Loss: 1.0897
  train Batch 150/156 | Loss: 1.0520
  val Batch 50/156 | Loss: 0.9866
  val Batch 100/156 | Loss: 1.0209
  val Batch 150/156 | Loss: 1.0173


### todo 3

In [135]:
project_root = current_dir.parent.parent.parent
checkpoint_save3 = project_root / "notebooks" / "phase2notes" / "2_4_training_minior_topics" / "checkpoints" / "todo3"
checkpoint_save3.mkdir(parents=True, exist_ok=True)
checkpoint_save3 = checkpoint_save3 / "checkpoint.pt"

In [136]:
loaded_model2 = SpeakerCounter().to(device)
optimizer2 = optim.Adam(loaded_model2.parameters(), lr=0.0001)
scheduler2 = ReduceLROnPlateau(optimizer2, mode='min', patience=2, factor=0.5, min_lr=1e-6)

checkpoint_loaded2 = torch.load(checkpoint_save2)

loaded_model2.load_state_dict(checkpoint_loaded2['model_state_dict'])
optimizer2.load_state_dict(checkpoint_loaded2['optimizer_state_dict'])
scheduler2.load_state_dict(checkpoint_loaded2['scheduler_state_dict'])
start_epoch = checkpoint_loaded2['epoch'] + 1
best_val_loss = checkpoint_loaded2['val_loss']

In [137]:
print(f"last epoch it was on: {start_epoch}")
print(f"val loss achieved: {best_val_loss}")

last epoch it was on: 2
val loss achieved: 0.8875212415008787


In [138]:
num_epochs = 2

for epoch in range(num_epochs):
    loaded_model2.train()
    running_train_loss = 0.0
    running_train_acc = 0.0
    steps_per_epoch = len(train_loader)
    
    for step, batch in enumerate(train_loader):
        features, labels = batch[0].to(device), batch[1].to(device)
        
        # Forward pass
        logits = loaded_model2(features)  
        
        # Compute loss
        loss = F.cross_entropy(logits, labels)
        
        # Backward pass
        loss.backward()
        
        # Update weights
        optimizer2.step()
        optimizer2.zero_grad()
        
        # Compute accuracy
        running_train_loss += loss.item() 
        running_train_acc += compute_accuracy(logits, labels)
        
        # Optional: Print batch progress every 50 batches within the epoch
        if (step + 1) % 50 == 0:
            print(f"  train Batch {step+1}/{steps_per_epoch} | Loss: {loss.item():.4f}")
    
    # 3. Print Summary at the end of each FULL epoch
    avg_train_loss = running_train_loss / steps_per_epoch
    avg_train_acc = running_train_acc / steps_per_epoch    
    
    loaded_model2.eval()
    running_val_loss = 0.0
    running_val_acc = 0.0      
    
    with torch.no_grad():
        for steps, batch in enumerate(val_loader):
            
            features, labels = batch[0].to(device), batch[1].to(device)
            
            # Forward pass
            logits = loaded_model2(features)  
            
            # Compute loss
            loss = F.cross_entropy(logits, labels)
            
            # number of correct class
            running_val_loss += loss.item()
            running_val_acc += compute_accuracy(logits, labels)  
            
            # Optional: Print batch progress every 50 batches within the epoch
            if (steps + 1) % 50 == 0:
                print(f"  val Batch {steps+1}/{steps_per_epoch} | Loss: {loss.item():.4f}")
    
    # Calculate Averages
    avg_val_loss = running_val_loss / len(val_loader)
    avg_val_acc = running_val_acc / len(val_loader)    
    
    # --- LOGGING & SCHEDULER ---
    print(f"\n--- Epoch {epoch+1}/{num_epochs} Summary ---")
    print(f"Train Loss: {avg_train_loss:.4f} | Train Acc: {avg_train_acc:.2f}%")
    print(f"Val Loss:   {avg_val_loss:.4f} | Val Acc:   {avg_val_acc:.2f}%")
    
    scheduler2.step(avg_val_loss)
    
    # Optional: Print current learning rate
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Current Learning Rate: {current_lr}\n")
    
    if avg_val_loss < best_val_loss:
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': loaded_model2.state_dict(),
            'optimizer_state_dict': optimizer2.state_dict(),
            'scheduler_state_dict': scheduler2.state_dict(),
            'val_loss': avg_val_loss,
            'val_acc': avg_val_acc,
        }
        torch.save(checkpoint, checkpoint_save3)
        print(f"new best! Saved CheckPoint (loss:{avg_val_loss:.6f})")
    else:
        print("No improvement")




  train Batch 50/156 | Loss: 0.8766
  train Batch 100/156 | Loss: 0.8384
  train Batch 150/156 | Loss: 0.8714
  val Batch 50/156 | Loss: 0.8352
  val Batch 100/156 | Loss: 0.8752
  val Batch 150/156 | Loss: 0.8696

--- Epoch 1/2 Summary ---
Train Loss: 0.8698 | Train Acc: 53.80%
Val Loss:   0.8664 | Val Acc:   50.68%
Current Learning Rate: 0.0001

new best! Saved CheckPoint (loss:0.866350)
  train Batch 50/156 | Loss: 0.8618
  train Batch 100/156 | Loss: 0.8311
  train Batch 150/156 | Loss: 0.8707
  val Batch 50/156 | Loss: 0.8209
  val Batch 100/156 | Loss: 0.8530
  val Batch 150/156 | Loss: 0.8500

--- Epoch 2/2 Summary ---
Train Loss: 0.8473 | Train Acc: 53.51%
Val Loss:   0.8461 | Val Acc:   51.17%
Current Learning Rate: 0.0001

new best! Saved CheckPoint (loss:0.846119)
